# unbox-args-tensor-to-array — faded example 3: Recurse one level into containers

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `unbox-args-tensor-to-array`. The last cell reports your progress on the `Backprop: Unbox Tensor args to array` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Unbox Tensor args to array` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`unbox-args-tensor-to-array`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "unbox-args-tensor-to-array"
DD_SUBTOPIC = "Backprop: Unbox Tensor args to array"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Unboxing for `cat`/`stack` must recurse one level into a list/tuple arg, unwrapping inner `MiniTensor`s while preserving the container type. A `list` stays a `list`; a `tuple` stays a `tuple`.

## Faded exercise 3

### Unbox inside a list argument

Implement `_maybe_unbox(a)`, the per-value helper: unbox a `MiniTensor`, recurse one level into a `list` (preserving list type), and pass everything else through. The MiniTensor and pass-through cases are written; complete the `list` branch.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
class MiniTensor:
    def __init__(self, array):
        self.array = array

def _maybe_unbox(a):
    if isinstance(a, MiniTensor):
        return a.array
    if isinstance(a, list):
        raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above
    return a

m1, m2 = MiniTensor([1]), MiniTensor([2])
print(_maybe_unbox([m1, 5, m2]))


def _test():
    m1 = MiniTensor([1, 2])
    m2 = MiniTensor([3, 4])
    out = _maybe_unbox([m1, 5, m2])
    assert isinstance(out, list), type(out)
    assert out[0] is m1.array
    assert out[1] == 5
    assert out[2] is m2.array
    # top-level MiniTensor still unboxes
    assert _maybe_unbox(m1) is m1.array
    # plain scalar passes through
    assert _maybe_unbox(7) == 7


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class MiniTensor:
    def __init__(self, array):
        self.array = array

def _maybe_unbox(a):
    if isinstance(a, MiniTensor):
        return a.array
    if isinstance(a, list):
        return [_maybe_unbox(x) for x in a]
    return a

m1, m2 = MiniTensor([1]), MiniTensor([2])
print(_maybe_unbox([m1, 5, m2]))
```
</details>